<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
OpenAI Agent Builder
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
from genai_lib.utilities import mprint, mermaid

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---

***Multi-Agent Projekt*** hat gezeigt, wie man ein Multi-Agent-System **von Hand in LangGraph** baut.
**Dieses Modul** zeigt den anderen Weg: den **OpenAI Agent Builder** – ein visuelles No-Code-Werkzeug,
mit dem dieselben Workflows per Drag & Drop entworfen werden können.

Außerdem schauen wir uns den **OpenAI Agents SDK** an – die Python-Entsprechung
des visuellen Builders für Entwickler.

| Thema | M21 (LangGraph) | M22 (Agent Builder / SDK) |
|-------|-----------------|---------------------------|
| **Ansatz** | Code-first | No-Code + SDK |
| **Workflows** | Python StateGraph | Drag & Drop / `Agent` Klasse |
| **Tools** | `@tool` Decorator | Tool-Nodes / `@function_tool` |
| **Routing** | `supervisor_router()` | Condition-Nodes / LLM entscheidet |
| **Deployment** | Manuell | One-Click / Built-in |
| **Lernkurve** | Mittel | Niedrig (UI) / Gering (SDK) |


In [ ]:
#@markdown   <p><font size="4" color='green'>  Kurs-Einordnung</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart LR
    M19["M19\nMulti-Agent\nPatterns"]
    M20["M20\nSupervisor\nDeep Dive"]
    M21["M21\nMulti-Agent\nProjekt"]
    M29(["🎯 M29\nAgent Builder\n& SDK"])
    M30["M30\n..."]

    M19 --> M20 --> M21 --> M29 --> M30

    style M19  fill:#37474F,color:#fff
    style M20  fill:#37474F,color:#fff
    style M21  fill:#37474F,color:#fff
    style M29  fill:#FF9800,color:#000
    style M30  fill:#37474F,color:#fff
'''
mermaid(diagram, width=900)


# 2 | Was ist der Agent Builder?
---

Der **OpenAI Agent Builder** ist ein visuelles No-Code-Werkzeug zur Erstellung
komplexer KI-Workflows – vorgestellt auf dem OpenAI DevDay 2025 als Teil von **AgentKit**.

> ⚠️ **Voraussetzung:** ChatGPT **Enterprise** oder **Edu** Account  
> **URL:** [platform.openai.com/agent-builder](https://platform.openai.com/agent-builder)  
> ChatGPT Plus/Team hat **keinen** Zugang zum Agent Builder.

**Kernfunktionen auf einen Blick:**  
Drag & Drop Workflows · Bedingte Logik · Multi-Agent-Koordination · MCP-Integration (100+ Services) · Built-in Monitoring & Versionierung · Code-Export (TypeScript/Python)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Agent Builder Interface-Bereiche</font> </br></p>

diag_iface = '''
%%{init: {'theme':'forest'}}%%
stateDiagram-v2
    [*] --> Templates : Start hier
    Templates --> Drafts : Anpassen
    Drafts --> Drafts : Iterieren
    Drafts --> Workflows : Veröffentlichen
    Workflows --> Drafts : Klonen / Bearbeiten
    Workflows --> [*] : Deployen
'''
mermaid(diag_iface, width=650)

<p><font color='black' size="5">Node-Typen</font></p>

Agent Builder arbeitet mit einem gerichteten Graphen aus **Nodes** (Aktionen)
und **Edges** (Verbindungen) – konzeptionell identisch mit LangGraph.

| Node-Typ | Symbol | Funktion | Beispiel |
|----------|--------|----------|----------|
| **LLM** | 🤖 | Modell-Aufruf mit Prompt | Klassifikation, Zusammenfassung |
| **Tool** | 🔧 | API-Call oder MCP-Server | JIRA-Ticket, E-Mail senden |
| **Condition** | 🔀 | Verzweigung auf Basis von Daten | Wenn Priorität > 3, dann… |
| **Human** | 👤 | Human-in-the-Loop Checkpoint | Genehmigung einholen |
| **Subworkflow** | 📦 | Verschachtelung anderer Workflows | Wiederverwendbare Sub-Prozesse |

In [ ]:
#@markdown   <p><font size="4" color='green'>  Node-Typen im Überblick</font> </br></p>

diag_nodes = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph NT [Node-Typen im Agent Builder]
        A["🤖 LLM Node"]
        B["🔧 Tool Node"]
        C["🔀 Condition Node"]
        D["👤 Human Node"]
        E["📦 Subworkflow Node"]
    end

    A -->|Text analysieren| OUT["Output / nächster Node"]
    B -->|Externe Aktion| OUT
    C -->|Routing-Logik| OUT
    D -->|Genehmigung| OUT
    E -->|Komplex-Logik| OUT

    style A fill:#2196F3,color:#fff
    style B fill:#FF9800,color:#000
    style C fill:#FDD835,color:#000
    style D fill:#4CAF50,color:#fff
    style E fill:#9C27B0,color:#fff
    style OUT fill:#37474F,color:#fff
'''
mermaid(diag_nodes, width=900)

# 3 | No-Code vs. Code-Ansatz
---

Für Multi-Agent-Workflows gibt es drei Ansätze – von visuell bis code-nah:

| | Agent Builder | OpenAI Agents SDK | LangGraph |
|---|---|---|---|
| **Coding erforderlich** | ❌ | ✅ Python | ✅ Python |
| **Schnelles Prototyping** | ✅ | ✅ | ⚠️ |
| **Volle Code-Kontrolle** | ⚠️ Export | ✅ | ✅ |
| **Multi-Provider (OpenAI + Anthropic)** | ❌ | ❌ | ✅ |
| **On-Premise Deployment** | ❌ | ✅ | ✅ |
| **Built-in Monitoring** | ✅ | ⚠️ | ⚠️ LangSmith |
| **Built-in Versionierung** | ✅ | ❌ | ❌ |
| **MCP-Integration** | ✅ Native | ✅ | ⚠️ Custom |
| **Human-in-the-Loop** | ✅ Node | ✅ | ✅ M19 |
| **Lernkurve** | Niedrig | Gering | Mittel |
| **Kosten (Entwicklung)** | Niedrig | Niedrig | Mittel |

> ⚠️ Code-Export aus dem Builder möglich, aber auf TypeScript/Python limitiert

**Reasoning-Modelle** erweitern das Bild: Unabhängig vom Framework stellt sich die Frage, welches **Modell** im Agenten steckt.

| | Standard-Modell | Reasoning-Modell |
|---|---|---|
| **Beispiele** | gpt-4o-mini, gpt-4o | o3-mini, o3, Claude 3.7 (extended thinking) |
| **Arbeitsweise** | Antwort direkt | Internes "Denken" vor Antwort (Chain of Thought) |
| **Staerke** | Schnell, kostenguenstig | Komplexe Logik, mehrstufige Planung |
| **Latenz** | Niedrig | Hoeher (Thinking-Phase) |
| **Kosten** | Gering | 3–10× hoeher |
| **Einsatz im Agenten** | Einfaches Routing, Extraktion, FAQ | Supervisor-Logik, Code-Debugging, Mathematik |

**Wann lohnt ein Reasoning-Modell?**
- Supervisor-Agent mit vielen Worker-Agents und komplexen Routing-Bedingungen (*Supervisor Pattern*/*Multi-Agent Projekt*)
- Tool-Auswahl bei 5+ Tools mit feinen Bedeutungsunterschieden (*Multi-Tool Agents*)
- Aufgaben, die mehrere Planungsschritte erfordern, bevor ein Tool aufgerufen wird

> **Framework-unabhaengig:** `init_chat_model("openai:o3-mini")` tauscht das Modell aus – der Graph-Code in LangGraph oder der Agent-Code bleibt unveraendert.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsbaum – Builder vs. SDK vs. LangGraph</font> </br></p>

diag_ent = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START(["❓ Ich baue einen Agenten"]) --> Q1{"Team kann Python\nprogrammieren?"}

    Q1 -->|"NEIN"| AB["✅ Agent Builder\nNo-Code Workflows"]
    Q1 -->|"JA"| Q2{"Nur OpenAI-Modelle\nausreichend?"}

    Q2 -->|"JA"| Q3{"Einfacher Agent\nohne State Machine?"}
    Q2 -->|"NEIN\n(Multi-Provider)"| LG["✅ LangGraph\n(M12–M22)"]

    Q3 -->|"JA"| SDK["✅ OpenAI Agents SDK\nPython, einfach"]
    Q3 -->|"NEIN\n(komplexe Zustände)"| LG

    AB --- N1["Drag & Drop\nMCP-Integration\nEnterprise Governance"]
    SDK --- N2["@function_tool\nHandoffs\nStreamlit-fähig"]
    LG  --- N3["StateGraph\nCheckpointing\nSupervisor Pattern"]

    style AB  fill:#10a37f,color:#fff
    style SDK fill:#FF9800,color:#000
    style LG  fill:#2196F3,color:#fff
    style N1  fill:#1B5E20,color:#fff
    style N2  fill:#E65100,color:#fff
    style N3  fill:#0D47A1,color:#fff
'''
mermaid(diag_ent, width=520)

# 4 | Live-Demo: Support-Ticket-Routing
---

**Szenario:** Eingehende Support-Tickets werden automatisch kategorisiert,
priorisiert und an die richtige Abteilung weitergeleitet.

**Anforderungen:**
- Kategorisierung: `technical` | `billing` | `sales`
- Prioritäts-Bewertung: 1–5 (5 = kritisch)
- Bedingte Weiterleitung je nach Kategorie & Priorität
- Human-in-the-Loop für Sales-Anfragen
- Bestätigungs-E-Mail an Kunden

In [ ]:
#@markdown   <p><font size="4" color='green'>  Support-Ticket-Routing Workflow</font> </br></p>

diag_ticket = '''
%%{init: {'theme':'dark'}}%%
flowchart TB
    START(["📨 Ticket eingehend"]) --> PARSE["🔧 Parse Ticket Data"]
    PARSE --> LLM["🤖 LLM: Analyze & Categorize\ngpt-4o-mini, temp=0.0"]

    LLM --> COND{"🔀 Category + Priority?"}

    COND -->|"technical\nPriority > 3"| JIRA["🔧 Create JIRA Ticket"]
    COND -->|"technical\nPriority ≤ 3"| QUEUE["🔧 Add to Support Queue"]
    COND -->|"billing"| FINANCE["🔧 Assign to Finance"]
    COND -->|"sales"| HUMAN["👤 Human Review Required"]

    JIRA --> EMAIL["🔧 Send Confirmation Email"]
    QUEUE --> EMAIL
    FINANCE --> EMAIL
    HUMAN --> APPR{"Approved?"}
    APPR -->|"Ja"| EMAIL
    APPR -->|"Nein"| REJECT["❌ Send Rejection Notice"]

    EMAIL --> FINISH(["✅ Workflow Complete"])
    REJECT --> FINISH

    style START  fill:#2E7D32,color:#fff
    style FINISH fill:#1A237E,color:#fff
    style LLM    fill:#2196F3,color:#fff
    style COND   fill:#FDD835,color:#000
    style HUMAN  fill:#4CAF50,color:#fff
    style APPR   fill:#FDD835,color:#000
    style REJECT fill:#B71C1C,color:#fff
    style JIRA   fill:#FF9800,color:#000
    style QUEUE  fill:#FF9800,color:#000
    style FINANCE fill:#FF9800,color:#000
    style EMAIL  fill:#FF9800,color:#000
'''
mermaid(diag_ticket, width=820)

<p><font color='black' size="5">Node-Konfigurationen (Konzept)</font></p>

Im Agent Builder wird jeder Node über ein Formular konfiguriert. Die drei Kern-Nodes des Support-Workflows:

> **🤖 LLM Node `Analyze & Categorize`**  
> Modell: `gpt-4o-mini` · Temperature: `0.0`  
> Klassifiziert Ticket → `technical | billing | sales` + Priorität 1–5  
> Input: `{ticket_text}` → Output: JSON `{category, priority, summary}`

> **🔀 Condition Node `Category Router`**  
> `technical + Prio > 3` → JIRA · `technical + Prio ≤ 3` → Queue · `billing` → Finance · `sales` → Human Review

> **🔧 Tool Node `Create JIRA Ticket` (via MCP)**  
> Verbindet sich mit dem JIRA-MCP-Server, ruft `create_issue` auf  
> Input: `{summary}`, `{priority}` → Output: JIRA-ID

**Das LangGraph-Äquivalent** dieser drei Nodes ist in Kapitel 5 implementiert und vollständig ausführbar.

In [ ]:
#@markdown   <p><font size="4" color='green'>  MCP-Integration im Workflow</font> </br></p>

diag_mcp = '''
%%{init: {'theme':'dark'}}%%
flowchart TB
    WF["Agent Builder Workflow"] --> MCP["MCP Protocol Layer"]

    MCP --> JIRA["JIRA Server"]
    MCP --> SLACK["Slack Server"]
    MCP --> EMAIL2["E-Mail Server"]
    MCP --> DB["PostgreSQL Server"]
    MCP --> CUSTOM["Custom MCP Server"]

    JIRA   --> JAPI["JIRA API"]
    SLACK  --> SAPI["Slack API"]
    EMAIL2 --> EAPI["SMTP / SendGrid"]
    DB     --> DAPI["Database"]
    CUSTOM --> CAPI["Ihre API"]

    style MCP    fill:#10a37f,color:#fff
    style WF     fill:#2196F3,color:#fff
    style JIRA   fill:#FF9800,color:#000
    style SLACK  fill:#9C27B0,color:#fff
    style EMAIL2 fill:#FF9800,color:#000
    style CUSTOM fill:#37474F,color:#fff
'''
mermaid(diag_mcp, width=780)

# 5 | Vergleich: Builder vs. LangGraph
---

Der Support-Ticket-Workflow aus Kapitel 4 lässt sich 1:1 in LangGraph nachbauen.
Der Vergleich zeigt: **gleiche Konzepte, andere Werkzeuge**.

<p><font color='black' size="5">Konzept-Äquivalenzen</font></p>

| Agent Builder | LangGraph / LangChain | Funktion |
|---------------|----------------------|----------|
| LLM Node | `llm.invoke()` in Node-Funktion | Modell aufrufen |
| Condition Node | `add_conditional_edges()` + Router | Bedingtes Routing |
| Tool Node | `@tool` Decorator | Externe Aktion |
| Human Node | `interrupt()` aus M19 | Human-in-the-Loop |
| Subworkflow | Nested `StateGraph` | Wiederverwendbare Logik |
| Workflow veröffentlichen | `graph.compile()` | Graph fertigstellen |
| MCP Server | `@tool` mit HTTP-Request | Service-Integration |
| Versioning | Git-Commits | Nachvollziehbarkeit |

In [ ]:
#@markdown   <p><font size="4" color='green'>  Builder-Konzepte in LangGraph</font> </br></p>

diag_equiv = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph AB ["Agent Builder (No-Code)"]
        AB1["🤖 LLM Node"]
        AB2["🔀 Condition Node"]
        AB3["🔧 Tool Node"]
        AB4["👤 Human Node"]
    end

    subgraph LG ["LangGraph (Python)"]
        LG1["llm.invoke()\nin Node-Funktion"]
        LG2["add_conditional_edges()\n+ Router-Funktion"]
        LG3["@tool Decorator\n+ ReAct Agent"]
        LG4["interrupt()\n(M18 HITL)"]
    end

    AB1 <-."gleichwertig".-> LG1
    AB2 <-."gleichwertig".-> LG2
    AB3 <-."gleichwertig".-> LG3
    AB4 <-."gleichwertig".-> LG4

    style AB1 fill:#10a37f,color:#fff
    style AB2 fill:#10a37f,color:#fff
    style AB3 fill:#10a37f,color:#fff
    style AB4 fill:#10a37f,color:#fff
    style LG1 fill:#2196F3,color:#fff
    style LG2 fill:#2196F3,color:#fff
    style LG3 fill:#2196F3,color:#fff
    style LG4 fill:#2196F3,color:#fff
'''
mermaid(diag_equiv, width=900)


# 6 | OpenAI Agents SDK — Referenz
---

Der **OpenAI Agents SDK** ist die Python-Entsprechung des visuellen Agent Builders.
Er erlaubt dieselben Konzepte – Agents, Tools, Handoffs – in reinem Python-Code.

**Vorteile gegenüber dem visuellen Builder:**
- ✅ Volle Code-Kontrolle
- ✅ In Jupyter, Skripten und Anwendungen nutzbar
- ✅ Custom Python-Logik in Tools
- ✅ Kein Enterprise-Account nötig

**Schlüsselkonzepte:**

| Konzept | Beschreibung |
|---------|-------------|
| `Agent` | Basisklasse – Modell + Instructions + Tools |
| `@function_tool` | Decorator für Python-Funktionen als Tools |
| `Runner.run()` | Führt einen Agenten aus (async) |
| **Handoffs** | Agent delegiert an anderen Agent |
| `RunResult` | Enthält alle Schritte und die finale Antwort |

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: OpenAI Agents SDK Architektur</font> </br></p>

diag_sdk = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    USER["👤 User-Anfrage"] --> RUNNER["Runner.run(agent, input)"]

    RUNNER --> AGENT["🤖 Agent\ninstructions + model"]

    AGENT --> THINK{"Tool nötig?"}
    THINK -->|"JA"| TOOL["@function_tool\nPython-Funktion"]
    THINK -->|"NEIN"| FINAL["Finale Antwort"]
    TOOL --> OBS["Tool-Ergebnis\nbeobachten"]
    OBS --> THINK

    AGENT --> HANDOFF{"Handoff?"}
    HANDOFF -->|"JA"| AGENT2["🤖 Spezialist-Agent"]
    AGENT2 --> FINAL

    FINAL --> OUT["RunResult\n(Schritte + Antwort)"]

    style AGENT   fill:#10a37f,color:#fff
    style AGENT2  fill:#10a37f,color:#fff
    style RUNNER  fill:#2196F3,color:#fff
    style TOOL    fill:#FF9800,color:#000
    style HANDOFF fill:#FDD835,color:#000
    style THINK   fill:#FDD835,color:#000
    style OUT     fill:#37474F,color:#fff
'''
mermaid(diag_sdk, width=550)

<p><font color='black' size="5">OpenAI Agents SDK vs. LangGraph</font></p>

| Merkmal | OpenAI Agents SDK | LangGraph |
|---------|-------------------|-----------|
| **Einstieg** | Niedrig | Mittel |
| **Provider** | Nur OpenAI | Multi-Provider |
| **State Management** | Implizit (intern) | Explizit (`TypedDict`) |
| **Routing** | LLM entscheidet (Handoffs) | Code-basiert (Router-Funktion) |
| **Human-in-the-Loop** | Über Hooks | `interrupt()` |
| **Debugging** | `result.new_messages` | LangSmith Tracing |
| **Deployment** | Eigene Infrastruktur | Eigene Infrastruktur |
| **Lernkurve** | Gering | Mittel |
| **Wann nutzen** | Einfache OpenAI-only Workflows | Komplexe / Multi-Provider-Systeme |

> **Fazit:** Für diesen Kurs (LangGraph, Multi-Provider, volle Kontrolle) bleibt LangGraph die richtige Wahl.  
> Das Agents SDK ist eine gute Alternative für kleinere Projekte, die ausschließlich OpenAI nutzen.

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
Aufgabe 1 — Support-Ticket-Routing im Agent Builder nachbauen
</font></p>

Baue den Workflow aus Kapitel 4 im Agent Builder nach:

1. **LLM Node** `Analyze & Categorize` — System Prompt schreiben: Ticket klassifizieren in `technical | billing | sales`, Priorität 1–5
2. **Condition Node** `Category Router` — Routing-Logik definieren:
   - `technical + Prio > 3` → JIRA-Tool
   - `technical + Prio ≤ 3` → Support Queue
   - `billing` → Finance
   - `sales` → Human Review
3. **Tool Nodes** für JIRA und Support Queue verbinden
4. **Human Node** für Sales-Anfragen ergänzen

> 💡 Referenz: Kap. 4 — Mermaid-Diagramm zeigt den vollständigen Workflow

<p><font color='black' size="5">
Aufgabe 2 — Eigenen Workflow im Agent Builder entwerfen
</font></p>

Wähle einen eigenen Use Case und baue den Workflow im Agent Builder:

**Mindestanforderungen:**
- 1 LLM Node (mit eigenem System Prompt)
- 1 Condition Node (Routing-Logik)
- 1 Tool Node (MCP-Server oder Custom Tool)
- 1 Human Node (für kritische Entscheidungen)

**Ideen:**
| Use Case | LLM Node | Condition | Tool | Human |
|----------|----------|-----------|------|-------|
| Bewerbungs-Screener | Bewerbung analysieren | Qualifiziert? | ATS-Eintrag | Manager-Review |
| Content-Moderator | Inhalt prüfen | Verstoß? | Slack-Alert | Redaktion |
| Reise-Planer | Wunsch analysieren | Budget? | Hotel-/Flug-API | Buchung |

> 💡 **Tipp:** Starte mit dem Entscheidungsbaum aus Kap. 3 — hilft bei der Wahl zwischen Builder, Agents SDK und LangGraph

<p><font color='black' size="5">
Aufgabe 3: Builder vs. LangGraph – Eigene Entscheidung
</font></p>

Wähle **einen** der folgenden Anwendungsfälle und entscheide:
Würdest du Agent Builder, den Agents SDK oder LangGraph verwenden?

**Szenarien:**

| Szenario | Beschreibung |
|----------|--------------|
| A | Ein Marketing-Team (kein Coding) will Blogposts automatisch in Social-Media-Posts umwandeln |
| B | Eine Bank braucht On-Premise-Deployment für einen Kredit-Prüf-Agenten |
| C | Ein Startup will schnell einen MVP für einen Kunden-Support-Bot bauen |
| D | Ein Data-Science-Team will mehrere LLM-Provider vergleichen (OpenAI vs. Anthropic) |

**Aufgabe:**
1. Wähle ein Szenario
2. Begründe deine Entscheidung (Werkzeug + Warum) in einer Markdown-Zelle
3. Skizziere die Architektur mit einem Mermaid-Diagramm
4. Liste die 3 wichtigsten Vorteile deines gewählten Ansatzes auf